# Gjøvik Space Agency

## Forsyningskjeden til Månebase Mjøsa

### Pilotprosjekt for Matematikk 1, logistikk

Gjøvik Space Agency skal forsyne en permanent månebase med vann, mat, reservedeler og teknisk utstyr. Forsyningskjeden består av fire ledd:

```text
Gjøvik produksjon
        ↓
oppskytningsbase
        ↓
depot i månebane
        ↓
månelander
        ↓
Månebase Mjøsa
```

Transporten har lange ledetider, begrenset kapasitet og perioder der enkelte ledd ikke er tilgjengelige. Et orbitalt depot kan gi større robusthet, men koster masse, oppdrag og lagerkapasitet.

Prosjektet har fem deler:

1. **Stasjonær forsyningskjede:** massebalanse og nødvendige aktivitetsrater
2. **Ett kritisk lager:** en tokomponents lager- og responsmodell
3. **Hele kjeden:** lager og transportkapasitet som en vektor-ODE
4. **Rommekanikk:** transporttid, $\Delta v$ og nyttelastkapasitet
5. **Egenmoder:** stabilitet, langsomme responsmoder og virkningen av depot

### Læringsmål

Etter prosjektet skal du kunne

- bygge en forsyningsmatrise fra leveranser og forbruk,
- finne en stasjonær flyt med et lineært system,
- kontrollere massebalanse og kapasitet,
- skille mellom kontinuerlig gjennomsnittsstrøm og partivise oppdrag,
- formulere en lager- og responsmodell som en vektor-ODE,
- bruke Euler på en hel forsyningskjede,
- modellere transportavbrudd med en tidsavhengig tilgjengelighetsmatrise,
- beregne idealisert Hohmann-tid og $\Delta v$,
- bruke rakettligningen til å anslå nyttelastkapasitet,
- linearisere omkring en likevekt,
- beregne og tolke egenverdier og egenvektorer,
- diskutere sikkerhetslager, robusthet og gjenoppretting.

### Modellavgrensning

Modellen er deterministisk og bruker kontinuerlige gjennomsnittsstrømmer. Virkelige oppskytninger er diskrete, transporttidene varierer, og romfartøy har mange flere masse-, sikkerhets- og banebegrensninger. Rommekanikken er idealisert og skal brukes til å bestemme pedagogiske ledetider og kapasiteter, ikke til oppdragsplanlegging.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SECONDS_PER_DAY = 24*3600.0
DAYS_PER_YEAR = 365.25

# Referanseoppdraget

Månebasen har seks personer. Vi samler lasten i tre varegrupper:

1. **forbruksvarer**: vann, mat, oksygen og hygiene
2. **reservedeler**: mekaniske og elektriske komponenter
3. **teknisk last**: forsøksutstyr, batterier og oppgraderinger

Basens gjennomsnittlige behov settes til:

$$
d=
\begin{pmatrix}
180\\12\\18
\end{pmatrix}
\quad\mathrm{kg/døgn}.
$$

Tallene er scenarioverdier valgt for å gi et tydelig matematisk prosjekt.

In [ ]:
vare_navn = ["forbruksvarer", "reservedeler", "teknisk last"]
behov_base = np.array([180.0, 12.0, 18.0])  # kg/døgn

for navn, d in zip(vare_navn, behov_base):
    print(f"{navn:16s}: {d:6.1f} kg/døgn")
print("Samlet leveringsbehov:", np.sum(behov_base), "kg/døgn")

# Del A: Stasjonær masseflyt

## A.1 Aktiviteter og leveringsgrader

Vi beskriver én samlet masseflyt gjennom fire aktiviteter:

1. klargjøring på Gjøvik
2. transport til og oppskyting fra oppskytningsbasen
3. transport og omlasting ved orbitalt depot
4. landing og lossing på månebasen

Hvert ledd har en leveringsgrad $\eta_j$. Den delen som ikke leveres videre kan representere emballasje, reserver, utrangert last eller andre tap i den aggregerte modellen.

Vi bruker:

$$
\eta_G=0.98,\quad
\eta_O=0.94,\quad
\eta_D=0.97,\quad
\eta_L=0.96.
$$

In [ ]:
eta = np.array([0.98, 0.94, 0.97, 0.96])
aktivitet_navn = ["Gjøvik", "oppskyting", "orbitalt depot", "månelander"]

## A.2 Lineært system for aktivitetsratene

La

$$
\mu=
\begin{pmatrix}
\mu_G\\\mu_O\\\mu_D\\\mu_L
\end{pmatrix}
$$

måles i kg/døgn. Stasjonær balanse krever

$$
\eta_G\mu_G-\mu_O=0,
$$

$$
\eta_O\mu_O-\mu_D=0,
$$

$$
\eta_D\mu_D-\mu_L=0,
$$

$$
\eta_L\mu_L=d_{total}.
$$

Dette gir

$$
\boxed{A_\mu\mu=b_\mu.}
$$

## Oppgave A1: Finn de nødvendige aktivitetsratene

Bygg $A_\mu$ og løs systemet. Kontroller residualet.

In [ ]:
d_total = np.sum(behov_base)

A_mu = np.array([
    [..., ..., 0.0, 0.0],
    [0.0, ..., ..., 0.0],
    [0.0, 0.0, ..., ...],
    [0.0, 0.0, 0.0, ...]
], dtype=float)

b_mu = np.array([0.0, 0.0, 0.0, d_total])
mu_likevekt = ...

for navn, verdi in zip(aktivitet_navn, mu_likevekt):
    print(f"{navn:16s}: {verdi:7.2f} kg/døgn")
print("Residualnorm:", ...)

## Oppgave A2: Flere varer

Samme system brukes for hver varegruppe. Løs alle tre høyresidene samtidig:

$$
A_\mu M_\mu =B,
$$

slik at hver kolonne i $M_\mu$ gir aktivitetsratene for én varegruppe.

In [ ]:
B_varer = np.zeros((4, 3))
B_varer[3, :] = behov_base

M_varer = ...
print("Aktivitetsrater per varegruppe:
", M_varer)
print("Kontroll:
", ...)

## A.3 Kapasitet og antall oppdrag

Anta at én oppskyting kan levere $6200$ kg til kjedens første romledd etter nødvendige reserver. Dersom det gjennomføres $N$ oppdrag per år, blir gjennomsnittskapasiteten

$$
\boxed{\mu_{kap}=\frac{NM_{nyttelast}}{365.25}.}
$$

Den kontinuerlige modellen gir et minimumstall, men den virkelige frekvensen må være et helt antall oppdrag.

In [ ]:
nyttelast_per_oppdrag = 6200.0  # kg

minimum_oppdrag_kontinuerlig = mu_likevekt[1]*DAYS_PER_YEAR/nyttelast_per_oppdrag
minimum_oppdrag_heltall = int(np.ceil(minimum_oppdrag_kontinuerlig))

print("Kontinuerlig oppdragsbehov:", minimum_oppdrag_kontinuerlig)
print("Minste hele antall oppdrag:", minimum_oppdrag_heltall)

## Oppgave A3: Flaskehals og margin

Bruk årlige kapasiteter:

- Gjøvik: $120000$ kg/år
- oppskyting: $18$ oppdrag/år
- orbitalt depot: $90000$ kg/år
- månelander: $76000$ kg/år

Beregn kapasitetsutnyttelsen i alle ledd. Hva er flaskehalsen? Hva skjer dersom månelanderens kapasitet reduseres med 20 prosent?

In [ ]:
kapasitet_årlig = np.array([
    120000.0,
    18*nyttelast_per_oppdrag,
    90000.0,
    76000.0
])

utnyttelse = ...
for navn, u in zip(aktivitet_navn, utnyttelse):
    print(f"{navn:16s}: {100*u:6.1f} %")

# Del B: Ett kritisk lager

## B.1 Lager og responsrate

La $q(t)$ være lagerbeholdningen av forbruksvarer på månebasen og $\mu(t)$ ønsket leveringsrate.

$$
\boxed{\dot q=\mu-d.}
$$

Forsyningssystemet forsøker å holde lageret nær $q^*$:

$$
\boxed{T\dot\mu=d+k(q^*-q)-\mu.}
$$

$T$ representerer samlet treghet i planlegging, klargjøring, transport og lossing.

In [ ]:
d_kritisk = behov_base[0]  # kg/døgn
q_stjerne = 90*d_kritisk       # 90 døgn sikkerhetslager
T_respons = 24.0                # døgn
k_lager = 0.035                 # 1/døgn

## Oppgave B1: Likevekt

Vis at likevekten er

$$q=q^*,\qquad\mu=d.$$

Skriv systemet for avvikene

$$e=q-q^*,\qquad r=\mu-d$$

som

$$
\frac{d}{dt}
\begin{pmatrix}e\\r\end{pmatrix}
=A_2
\begin{pmatrix}e\\r\end{pmatrix}.
$$

In [ ]:
A_2 = np.array([
    [0.0, 1.0],
    [..., ...]
])

print("A_2 =
", A_2)
print("Egenverdier:", ...)

## B.2 En leveringsforsinkelse

Vi bruker først en enkel hendelse: leveringsraten faller momentant til 40 prosent i 18 døgn. Etter hendelsen forsøker kontrollsystemet å bygge lageret opp igjen.

Denne modellen bruker fortsatt én responsrate, ikke en eksplisitt tidsforsinkelsesligning.

In [ ]:
def lager_ode(t_døgn, x):
    q, mu = x
    tilgjengelighet = 0.40 if 30.0 <= t_døgn < 48.0 else 1.0
    levert = tilgjengelighet*mu
    dq = levert-d_kritisk
    dmu = (d_kritisk+k_lager*(q_stjerne-q)-mu)/T_respons
    return np.array([dq, dmu])


def euler_system(f, x0, sluttid, dt):
    n = int(round(sluttid/dt))
    t = np.linspace(0.0, n*dt, n+1)
    X = np.zeros((n+1, len(x0)))
    X[0] = x0
    for j in range(n):
        X[j+1] = ...
    return t, X

## Oppgave B2: Sikkerhetslager og aggressiv respons

Simuler 180 døgn fra likevekt. Sammenlign:

- 30, 60 og 90 døgn sikkerhetslager
- liten og stor $k$
- kort og lang responstid $T$
- kort og lang leveringsstans

Rapporter minste lager, største leveringsrate og tiden til lageret er tilbake innen 2 prosent av målverdien.

In [ ]:
x0_B = np.array([q_stjerne, d_kritisk])

# Simuler og plott q og mu.

## B.3 Bullwhip som sidefenomen

Dersom $k$ er stor og $T$ er lang, kan leveringsraten oversvinge kraftig etter en lagerforstyrrelse. Dette ligner bullwhip-effekten, men hovedspørsmålet her er robusthet:

- Hvor stort sikkerhetslager trengs?
- Hvor raskt bør systemet reagere?
- Hvor stor leveringskapasitet trengs under gjenoppbygging?

# Del C: Hele forsyningskjeden

## C.1 Fire lagerpunkter

La

$$
q=
\begin{pmatrix}
q_G\\q_O\\q_D\\q_M
\end{pmatrix}
$$

være lager på Gjøvik, oppskytningsbasen, det orbitale depotet og månebasen.

Aktivitetsratene er

$$
\mu=
\begin{pmatrix}
\mu_G\\\mu_O\\\mu_D\\\mu_L
\end{pmatrix}.
$$

Forsyningsmatrisen er

$$
S=
\begin{pmatrix}
1&-1&0&0\\
0&\eta_O&-1&0\\
0&0&\eta_D&-1\\
0&0&0&\eta_L
\end{pmatrix}.
$$

Den siste lagerbalansen får i tillegg baseforbruket $-d$.

In [ ]:
eta_O = eta[1]
eta_D = eta[2]
eta_L = eta[3]

S = np.array([
    [1.0, -1.0, 0.0, 0.0],
    [0.0, eta_O, -1.0, 0.0],
    [0.0, 0.0, eta_D, -1.0],
    [0.0, 0.0, 0.0, eta_L]
])

b_ekstern = np.array([0.0, 0.0, 0.0, -d_total])

## Oppgave C1: Kontroller likevekten

Bruk aktivitetsraten fra del A og kontroller

$$
\boxed{S\bar\mu+b=0.}
$$

In [ ]:
print("Likevektsresidual:", ...)

## C.2 Lagerstyring i hvert ledd

Vi bruker ønskede lagre tilsvarende omtrent:

- Gjøvik: 25 døgn gjennomstrømning
- oppskytningsbase: 18 døgn
- orbitalt depot: 75 døgn
- månebase: 90 døgn

Hvert aktivitetsledd reagerer på lageret foran aktiviteten:

$$
\boxed{T\dot\mu=\bar\mu+K_q(q^*-q)-\mu.}
$$

Her er $T$ diagonal og inneholder responstidene.

In [ ]:
q_stjerne_kjede = np.array([
    25*mu_likevekt[1],
    18*mu_likevekt[2],
    75*mu_likevekt[3],
    90*d_total
])

T_kjede = np.diag([10.0, 18.0, 35.0, 12.0])
K_q = np.diag([0.025, 0.020, 0.012, 0.025])

## C.3 Tilgjengelighet og transportvinduer

Den faktiske aktiviteten er

$$
\mu_{faktisk}=A(t)\mu,
$$

med

$$A(t)=\operatorname{diag}(a_G,a_O,a_D,a_L).$$

I referansescenarioet:

- oppskytningsbasen er stengt fra døgn 70 til 85
- månelanderen er under vedlikehold fra døgn 125 til 135
- de andre leddene er tilgjengelige

In [ ]:
def tilgjengelighet(t_døgn):
    a = np.ones(4)
    if 70.0 <= t_døgn < 85.0:
        a[1] = 0.0
    if 125.0 <= t_døgn < 135.0:
        a[3] = 0.0
    return np.diag(a)

## C.4 Åttedimensjonalt ODE-system

Tilstanden er

$$
\boxed{X=(q_1,q_2,q_3,q_4,\mu_1,\mu_2,\mu_3,\mu_4)^T.}
$$

Ligningene er

$$
\boxed{\dot q=SA(t)\mu+b,}
$$

$$
\boxed{T\dot\mu=\bar\mu+K_q(q^*-q)-\mu.}
$$

## Oppgave C2: Implementer hele kjeden

Bruk `np.linalg.solve` på responstidsmatrisen $T$. Start i likevekt og simuler ett år.

In [ ]:
def kjede_ode(t_døgn, X):
    q = X[:4]
    mu = X[4:]
    A_t = tilgjengelighet(t_døgn)

    dq = ...
    rhs_mu = ...
    dmu = ...
    return np.concatenate([dq, dmu])

X0_C = np.concatenate([q_stjerne_kjede, mu_likevekt])

# Simuler 365 døgn og plott lagre og aktivitetsrater.

## Oppgave C3: Robusthet med og uten orbitalt depot

Sammenlign:

1. referansemodellen med 75 døgn lager i bane
2. 30 døgn lager i bane
3. nesten intet orbitalt sikkerhetslager
4. dobbelt depotlager
5. lengre oppskytningsstans

Rapporter:

- minste lager ved månebasen
- største aktivitetsrate i hvert ledd
- gjenopprettingstid
- samlet lagerbeholdning
- om noen lagre blir negative

# Del D: Rommekanikk setter ledetid og kapasitet

## D.1 Hohmann-transfer mellom to jordbaner

Vi bruker en illustrativ transport fra en parkeringsbane på 300 km til et depot i en sirkulær bane på 2000 km.

For sirkulære, koplanare baner:

$$
v_c(r)=\sqrt{\frac{\mu_E}{r}},
$$

$$a_t=\frac{r_1+r_2}{2},$$

$$
v_t(r)=\sqrt{\mu_E\left(\frac2r-\frac1{a_t}\right)}.
$$

Dermed:

$$\Delta v_1=v_t(r_1)-v_c(r_1),$$

$$\Delta v_2=v_c(r_2)-v_t(r_2),$$

$$
\boxed{t_H=\pi\sqrt{\frac{a_t^3}{\mu_E}}.}
$$

In [ ]:
mu_E = 3.986004418e14  # m^3/s^2
R_E = 6371e3            # m
r1 = R_E + 300e3
r2 = R_E + 2000e3

## Oppgave D1: Beregn overføringen

Beregn $\Delta v_1$, $\Delta v_2$, total $\Delta v$ og overføringstid.

In [ ]:
a_transfer = ...
v_c1 = ...
v_c2 = ...
v_t1 = ...
v_t2 = ...

delta_v1 = ...
delta_v2 = ...
delta_v_H = ...
t_H = ...

print("Delta-v 1:", delta_v1, "m/s")
print("Delta-v 2:", delta_v2, "m/s")
print("Samlet delta-v:", delta_v_H, "m/s")
print("Overføringstid:", t_H/3600, "timer")

## D.2 Rakettligningen og nyttelast

Rakettligningen er

$$
\boxed{\Delta v=I_{sp}g_0\ln\left(\frac{m_0}{m_f}\right).}
$$

Fartøyet har:

- maksimal startmasse i parkeringsbane: $18000$ kg
- tørrmasse: $6800$ kg
- spesifikk impuls: $450$ s
- manøverreserve: 20 prosent på toppen av Hohmann-$\Delta v$

Vi ser bort fra andre manøvrer i hovedberegningen.

In [ ]:
m0_maks = 18000.0
m_tørr = 6800.0
I_sp = 450.0
g0 = 9.80665
reservefaktor = 1.20

## Oppgave D2: Drivstoff og nyttelast

Finn masseforholdet, sluttmassen etter manøveren, drivstoffmassen og maksimal nyttelast.

Sammenlign med den oppgitte logistikkapasiteten på 6200 kg per oppdrag. Hva kan forklare forskjellen?

In [ ]:
delta_v_dim = reservefaktor*delta_v_H
masseforhold = ...
m_f = ...
drivstoff = ...
nyttelast_beregnet = ...

print("Dimensjonerende delta-v:", delta_v_dim)
print("Drivstoff:", drivstoff, "kg")
print("Beregnet nyttelast:", nyttelast_beregnet, "kg")

## D.3 Fra romferd til forsyningsmodell

Rommekanikken påvirker logistikkmodellen gjennom:

- kapasitet per oppdrag
- antall oppdrag per år
- reisetid
- reserve og emballasje
- tilgjengelighet

En mer fullstendig ledetid kan skrives

$$
\boxed{
T_{ledd}
=T_{planlegging}+T_{venting}+T_{reise}+T_{lossing}.}
$$

Hohmann-tiden er bare én del av den totale responstiden.

# Del E: Linearisering og egenmoder

## E.1 Konstant tilgjengelighet

Når $A(t)=I$, er systemet lineært omkring likevekten. Sett

$$\widetilde q=q-q^*,$$

$$\widetilde\mu=\mu-\bar\mu.$$

Da får vi

$$
\frac d{dt}
\begin{pmatrix}\widetilde q\\\widetilde\mu\end{pmatrix}
=
\mathcal A
\begin{pmatrix}\widetilde q\\\widetilde\mu\end{pmatrix},
$$

med

$$
\boxed{
\mathcal A=
\begin{pmatrix}
0&S\\
-T^{-1}K_q&-T^{-1}
\end{pmatrix}.}
$$

## Oppgave E1: Bygg systemmatrisen

Bruk blokkmatriser og kontroller at $\mathcal A$ er $8\times8$.

In [ ]:
T_inv = np.linalg.inv(T_kjede)
A_stor = np.block([
    [np.zeros((4, 4)), ...],
    [..., ...]
])

print("Form:", A_stor.shape)
print("Egenverdier:
", ...)

## Oppgave E2: Stabilitet og langsom mode

1. Finn egenverdiene.
2. Kontroller om alle realdeler er negative.
3. Finn egenverdien med realdel nærmest null.
4. Estimer den langsomste tidskonstanten som

$$
\tau\approx-\frac1{\operatorname{Re}\lambda}.
$$

5. Sammenlign systemet med stort og lite orbitalt depot.

Merk at lagerstørrelsen påvirker robustheten direkte, mens egenverdiene i denne enkle styringsloven hovedsakelig bestemmes av kobling, responsparametre og ledetider.

In [ ]:
egenverdier, egenvektorer = ...

# Identifiser langsomste stabile mode og tolk egenvektoren.

## E.3 Modal respons

Dersom systemmatrisen er diagonaliserbar, kan vi skrive

$$\mathcal A=PDP^{-1}$$

og bruke

$$X=Pz.$$

Sammenlign direkte Euler-simulering med en modal simulering for en liten lagerforstyrrelse.

In [ ]:
# Kontroller diagonaliserbarhet og implementer modal simulering.

# Valgfri romfartsfordypning: rendezvous

Den siste innflygingen til et orbitalt depot kan beskrives med en linearisert relativbevegelsesmodell nær en sirkulær mål-bane. Tilstanden kan være

$$X=(x,y,z,v_x,v_y,v_z)^T.$$

Clohessy–Wiltshire-modellen gir

$$
\dot X=A_{CW}X+Bu.
$$

Denne delen kan leveres som en ferdig funksjon. Den beregnede styringskostnaden

$$
\Delta v_{rendezvous}
\approx\sum_k\|u_k\|\Delta t
$$

kan legges til transfer- og reservebudsjettet. Dermed reduseres nyttelasten i forsyningsmodellen.

# Modellkritikk

Diskuter minst seks punkter:

- Gjennomsnittsstrømmene er kontinuerlige, mens oppdragene er diskrete.
- Leveringsgradene er konstante.
- Etterspørselen er deterministisk.
- Modellen skiller ikke alle varegruppene i den dynamiske delen.
- Transportavbrudd er forhåndsbestemte.
- Beredskapsoppdrag og kansellering modelleres ikke eksplisitt.
- Lagerstyringen er lineær og kan beordre urealistisk stor aktivitet.
- Aktivitetsratene har ingen eksplisitt metning i hoved-ODE-en.
- Negative lagre betyr mangel, ikke fysisk negativ beholdning.
- Hohmann-transferen antar to sirkulære og koplanare baner.
- Motorbrenningene antas impulsive.
- Rakettligningen beskriver bare det idealiserte masseforholdet.
- Måneoverføringen er ikke modellert direkte i hovedberegningen.
- Orbitale og bakkeoperative ledetider er sterkt forenklet.
- Egenverdianalysen gjelder nær likevekt med konstant tilgjengelighet.

## Mulige videreføringer

- flere varegrupper i den dynamiske modellen
- heltallsbegrensning på antall oppdrag
- stokastiske forsinkelser
- returlast og resirkulering
- flere månebaser
- prioritering ved mangel
- metning av produksjons- og transportrater
- orbitalt drivstoffdepot
- rendezvous som egen ODE-modul
- valg av besøksrekkefølge for flere satellitter

# Oppsummering

Skriv en kort rapport der du forklarer

1. hvordan stasjonær massebalanse ga et lineært system,
2. hvordan leveringsgrader påvirket nødvendig produksjon på Gjøvik,
3. forskjellen mellom gjennomsnittsrate og antall oppdrag,
4. hvordan sikkerhetslager og responstid påvirket ett kritisk lager,
5. hvordan hele kjeden ble skrevet som en vektor-ODE,
6. hvordan transportstans forplantet seg gjennom kjeden,
7. hvilken rolle det orbitale depotet spilte,
8. hvordan Hohmann-transferen ga transporttid og $\Delta v$,
9. hvordan rakettligningen koblet bane og nyttelast,
10. hva egenverdiene fortalte om stabilitet og gjenoppretting.

## Faglig bakgrunn

Forsyningsmodellen bygger på et kontinuerlig lager- og produksjonsrammeverk der leverings- og forbruksprosesser kobles gjennom en forsyningsmatrise. Rommekanikkdelen bruker idealiserte standardmodeller for Hohmann-transfer og rakettens masseforhold. Studentene trenger ikke eksterne kilder for hovedløpet.